In [1]:
import os
import numpy as np
import pandas as pd
import torch

# scikit-learn 버전 호환성 문제 해결을 위한 패치
# umap을 import하기 전에 패치를 적용해야 함
from sklearn.utils import check_array as sklearn_check_array
from sklearn.utils import validation

# check_array의 ensure_all_finite 파라미터를 처리하는 wrapper
def patched_check_array(*args, **kwargs):
    # ensure_all_finite 파라미터가 있으면 제거
    if 'ensure_all_finite' in kwargs:
        kwargs.pop('ensure_all_finite')
    return sklearn_check_array(*args, **kwargs)

# sklearn.utils.validation의 check_array를 패치
validation.check_array = patched_check_array

# umap을 import하기 전에 umap.umap_ 모듈의 check_array도 패치
import umap.umap_ as umap_module
umap_module.check_array = patched_check_array

# 이제 umap을 import
import umap

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from mmfe.core.model import MMFE, MMFEConfigs
from mmfe.utils.FeatureExtraction import MolEncoder

In [3]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [4]:
random_state = 2025

In [5]:
sns.set_theme(style='darkgrid')

# 1. Configuration

## Task Configuration

In [6]:
TASK_NAME = 'DIS'  # 'INH' or 'DIS'
START_FOLD = 1
END_FOLD = 5
VERSION = 'v15'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"[INFO] Task: {TASK_NAME}, Folds: {START_FOLD} to {END_FOLD}")

[INFO] Task: Kd, Folds: 1 to 5


In [7]:
# 2. UMAP 함수 정의
def doUMAP(embs, y, random_state):
    """UMAP을 사용하여 embedding을 2D로 축소"""
    # 입력 데이터를 float32 numpy array로 명시적으로 변환
    embs = np.asarray(embs, dtype=np.float32)
    
    # NaN이나 Inf 값이 있는지 확인하고 처리
    if np.any(np.isnan(embs)) or np.any(np.isinf(embs)):
        print("[WARNING] Found NaN or Inf values in embeddings, replacing with 0")
        embs = np.nan_to_num(embs, nan=0.0, posinf=0.0, neginf=0.0)
    
    # UMAP 적용
    # n_neighbors는 데이터 포인트 수보다 작아야 함
    n_neighbors = min(15, len(embs) - 1)
    if n_neighbors < 2:
        n_neighbors = 2
    
    # UMAP reducer 생성 및 fit
    reducer = umap.UMAP(
        n_components=2, 
        random_state=random_state, 
        n_neighbors=n_neighbors,
        min_dist=0.1
    )
    
    # fit 후 embedding_ 속성에서 결과 가져오기
    reducer.fit(embs)
    points = reducer.embedding_
    
    df_points = pd.DataFrame({
        'UMAP1': points[:, 0],
        'UMAP2': points[:, 1],
        'Ssel': y,
    })
    return df_points

# 3. Scatter plot 함수 정의
def scatterplot(df, filepath, title='Embedding Distribution'):
    """Ssel 값을 색상으로 사용한 scatter plot"""
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    
    scatter = ax.scatter(df['UMAP1'], df['UMAP2'], c=df['Ssel'], cmap='rainbow', alpha=0.6, s=20)
    ax.set_xlabel('UMAP1', fontsize=15)
    ax.set_ylabel('UMAP2', fontsize=15)
    ax.set_title(title, fontsize=16)
    
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Ssel', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()  # 메모리 절약을 위해 close
    print(f"[INFO] Saved plot to {filepath}")

In [8]:
def save_npy(
    emb_dir, X_train_val, y_train_val, X_test, y_test,
    emb_fp_train_val, emb_smi_train_val, emb_img_train_val,
    emb_fp_test, emb_smi_test, emb_img_test):
    
    np.save(
        os.path.join(emb_dir, f'fp_train_val_embeddings.npy'), 
        {
        'embeddings':  emb_fp_train_val,
        'Ssel': y_train_val,
        'SMILES': X_train_val,
        }
    )

    np.save(
        os.path.join(emb_dir, f'smi_train_val_embeddings.npy'), 
        {
        'embeddings': emb_smi_train_val,
        'Ssel': y_train_val,
        'SMILES': X_train_val,
        }
    )

    np.save(
        os.path.join(emb_dir, f'img_train_val_embeddings.npy'), 
        {
        'embeddings': emb_img_train_val,
        'Ssel': y_train_val,
        'SMILES': X_train_val,
        }
    )

    np.save(
        os.path.join(emb_dir, f'fp_test_embeddings.npy'), 
        {
        'embeddings': emb_fp_test,
        'Ssel': y_test,
        'SMILES': X_test,
        }
    )
    np.save(
        os.path.join(emb_dir, f'smi_test_embeddings.npy'), 
        {
        'embeddings': emb_smi_test,
        'Ssel': y_test,
        'SMILES': X_test,
        }
    )

    np.save(
        os.path.join(emb_dir, f'img_test_embeddings.npy'), 
        {
        'embeddings': emb_img_test,
        'Ssel': y_test,
        'SMILES': X_test,
        }
    )

In [9]:
# 4. 각 fold별로 처리
for fold_num in range(START_FOLD, END_FOLD + 1):
    print("\n" + "=" * 80)
    print(f"[Fold {fold_num}] Processing...")
    print("=" * 80)
    
    # 데이터 경로
    data_dir = f'./data/selectivity_processed/{TASK_NAME}/fold{fold_num}'
    
    # 데이터 로드
    print(f"[INFO] Loading data from {data_dir}")
    df_train = pd.read_csv(os.path.join(data_dir, 'selectivity_train.csv'))
    df_val = pd.read_csv(os.path.join(data_dir, 'selectivity_val.csv'))
    df_test = pd.read_csv(os.path.join(data_dir, 'selectivity_test.csv'))
    
    # Train과 Val 합치기
    df_train_val = pd.concat([df_train, df_val], ignore_index=True)
    
    print(f"Train: {len(df_train)}, Val: {len(df_val)}, Train+Val: {len(df_train_val)}, Test: {len(df_test)}")
    
    # SMILES와 Ssel 추출 (Train+Val과 Test 분리)
    X_train_val = df_train_val['SMILES'].values
    y_train_val = df_train_val['Ssel'].values.astype(np.float32)
    
    X_test = df_test['SMILES'].values
    y_test = df_test['Ssel'].values.astype(np.float32)
    
    # # MolEncoder 초기화
    # print("[INFO] Initializing MolEncoder...")
    # encoder = MolEncoder(DEVICE)
    # print("[INFO] MolEncoder initialized successfully")
    
    # # 각 modality별 embedding 생성 (Train+Val과 Test 분리)
    # print("[INFO] Generating embeddings for each modality...")
    
    # # FP (Fingerprint) features
    # print("  [FP] Extracting fingerprint features...")
    # emb_fp_train_val = np.array([encoder.get_fpt_features(smi) for smi in X_train_val], dtype=np.float32)
    # emb_fp_test = np.array([encoder.get_fpt_features(smi) for smi in X_test], dtype=np.float32)
    # print(f"  [FP] Train+Val embeddings shape: {emb_fp_train_val.shape}, Test embeddings shape: {emb_fp_test.shape}")
    
    # # SMI (SMILES) features
    # print("  [SMI] Extracting SMILES features...")
    # emb_smi_train_val = np.array([encoder.get_smi_features(smi) for smi in X_train_val], dtype=np.float32)
    # emb_smi_test = np.array([encoder.get_smi_features(smi) for smi in X_test], dtype=np.float32)
    # print(f"  [SMI] Train+Val embeddings shape: {emb_smi_train_val.shape}, Test embeddings shape: {emb_smi_test.shape}")
    
    # # IMG (Image) features
    # print("  [IMG] Extracting image features...")
    # emb_img_train_val = np.array([encoder.get_img_features(smi) for smi in X_train_val], dtype=np.float32)
    # emb_img_test = np.array([encoder.get_img_features(smi) for smi in X_test], dtype=np.float32)
    # print(f"  [IMG] Train+Val embeddings shape: {emb_img_train_val.shape}, Test embeddings shape: {emb_img_test.shape}")
    
    
    # --------------------------------------------------
    # Load fold-specific MMFE (must be trained/saved per fold)
    # --------------------------------------------------
    print("[INFO] Loading MMFE...")
    model_dir = os.path.join("./results", TASK_NAME, VERSION, "MMFE", f"fold{fold_num}")
    
    cfg = MMFEConfigs(
        fold_dir=model_dir,
        device=DEVICE,
        hidden_dim=32
    )
    
    mmfe = MMFE(cfg)
    mmfe.load()
    print("[INFO] MMFE loaded successfully")

    # --------------------------------------------------
    # Get pre-fusion modality embeddings (hidden descriptors)
    # --------------------------------------------------
    print("[INFO] Generating embeddings for each modality (pre-fusion descriptors)...")

    desc_train_val = mmfe.get_modality_descriptors(X_train_val)
    desc_test = mmfe.get_modality_descriptors(X_test)

    emb_fp_train_val = np.asarray(desc_train_val["fp"], dtype=np.float32)
    emb_smi_train_val = np.asarray(desc_train_val["smi"], dtype=np.float32)
    emb_img_train_val = np.asarray(desc_train_val["img"], dtype=np.float32)

    emb_fp_test = np.asarray(desc_test["fp"], dtype=np.float32)
    emb_smi_test = np.asarray(desc_test["smi"], dtype=np.float32)
    emb_img_test = np.asarray(desc_test["img"], dtype=np.float32)

    emb_dir = os.path.join(model_dir, 'embeddings')
    os.makedirs(emb_dir, exist_ok=True)    

    save_npy(
        emb_dir, X_train_val, y_train_val, X_test, y_test,
        emb_fp_train_val, emb_smi_train_val, emb_img_train_val,
        emb_fp_test, emb_smi_test, emb_img_test
    )
    

    print(f"  [FP]  Train+Val embeddings shape: {emb_fp_train_val.shape}, Test embeddings shape: {emb_fp_test.shape}")
    print(f"  [SMI] Train+Val embeddings shape: {emb_smi_train_val.shape}, Test embeddings shape: {emb_smi_test.shape}")
    print(f"  [IMG] Train+Val embeddings shape: {emb_img_train_val.shape}, Test embeddings shape: {emb_img_test.shape}")
    
    # UMAP 수행 및 시각화
    print("[INFO] Performing UMAP and visualization...")
    
    # 출력 디렉토리 생성
    figure_dir = os.path.join(f'{emb_dir}', 'UMAP')
    os.makedirs(figure_dir, exist_ok=True)
    
    # FP UMAP 및 시각화 (Train+Val과 Test 분리)
    df_fp_train_val_umap = doUMAP(emb_fp_train_val, y_train_val, random_state)
    df_fp_test_umap = doUMAP(emb_fp_test, y_test, random_state)
    scatterplot(df_fp_train_val_umap,
                os.path.join(figure_dir, f'fpt_train_val_embeddings.png'),
                title=f'Fingerprint Train+Val Embeddings Distribution (Fold {fold_num})')
    scatterplot(df_fp_test_umap,
                os.path.join(figure_dir, f'fpt_test_embeddings.png'),
                title=f'Fingerprint Test Embeddings Distribution (Fold {fold_num})')
    
    # SMI UMAP 및 시각화 (Train+Val과 Test 분리)
    df_smi_train_val_umap = doUMAP(emb_smi_train_val, y_train_val, random_state)
    df_smi_test_umap = doUMAP(emb_smi_test, y_test, random_state)
    scatterplot(df_smi_train_val_umap,
                os.path.join(figure_dir, f'smi_train_val_embeddings.png'),
                title=f'SMILES Train+Val Embeddings Distribution (Fold {fold_num})')
    scatterplot(df_smi_test_umap,
                os.path.join(figure_dir, f'smi_test_embeddings.png'),
                title=f'SMILES Test Embeddings Distribution (Fold {fold_num})')
    
    # IMG UMAP 및 시각화 (Train+Val과 Test 분리)
    df_img_train_val_umap = doUMAP(emb_img_train_val, y_train_val, random_state)
    df_img_test_umap = doUMAP(emb_img_test, y_test, random_state)
    scatterplot(df_img_train_val_umap,
                os.path.join(figure_dir, f'img_train_val_embeddings.png'),
                title=f'Image Train+Val Embeddings Distribution (Fold {fold_num})')
    scatterplot(df_img_test_umap,
                os.path.join(figure_dir, f'img_test_embeddings.png'),
                title=f'Image Test Embeddings Distribution (Fold {fold_num})')
    
    # 통계 정보 출력
    print(f"\n[INFO] Fold {fold_num} Statistics:")
    print(f"  FP Train+Val embeddings: shape={emb_fp_train_val.shape}, mean={emb_fp_train_val.mean():.4f}, std={emb_fp_train_val.std():.4f}")
    print(f"  FP Test embeddings: shape={emb_fp_test.shape}, mean={emb_fp_test.mean():.4f}, std={emb_fp_test.std():.4f}")
    print(f"  SMI Train+Val embeddings: shape={emb_smi_train_val.shape}, mean={emb_smi_train_val.mean():.4f}, std={emb_smi_train_val.std():.4f}")
    print(f"  SMI Test embeddings: shape={emb_smi_test.shape}, mean={emb_smi_test.mean():.4f}, std={emb_smi_test.std():.4f}")
    print(f"  IMG Train+Val embeddings: shape={emb_img_train_val.shape}, mean={emb_img_train_val.mean():.4f}, std={emb_img_train_val.std():.4f}")
    print(f"  IMG Test embeddings: shape={emb_img_test.shape}, mean={emb_img_test.mean():.4f}, std={emb_img_test.std():.4f}")
    print(f"  Train+Val Ssel: min={y_train_val.min():.4f}, max={y_train_val.max():.4f}, mean={y_train_val.mean():.4f}")
    print(f"  Test Ssel: min={y_test.min():.4f}, max={y_test.max():.4f}, mean={y_test.mean():.4f}")
    
    print(f"[Fold {fold_num}] Completed!")

print("\n" + "=" * 80)
print("[INFO] All folds processed!")
print("=" * 80)


[Fold 1] Processing...
[INFO] Loading data from ./data/selectivity_processed/Kd/fold1
Train: 399, Val: 45, Train+Val: 444, Test: 112
[INFO] Loading MMFE...
[INFO] MMFE loaded successfully
[INFO] Generating embeddings for each modality (pre-fusion descriptors)...
  [FP]  Train+Val embeddings shape: (444, 32), Test embeddings shape: (112, 32)
  [SMI] Train+Val embeddings shape: (444, 32), Test embeddings shape: (112, 32)
  [IMG] Train+Val embeddings shape: (444, 32), Test embeddings shape: (112, 32)
[INFO] Performing UMAP and visualization...
[INFO] Saved plot to ./results/Kd/v15/MMFE/fold1/embeddings/UMAP/fpt_train_val_embeddings.png
[INFO] Saved plot to ./results/Kd/v15/MMFE/fold1/embeddings/UMAP/fpt_test_embeddings.png
[INFO] Saved plot to ./results/Kd/v15/MMFE/fold1/embeddings/UMAP/smi_train_val_embeddings.png
[INFO] Saved plot to ./results/Kd/v15/MMFE/fold1/embeddings/UMAP/smi_test_embeddings.png
[INFO] Saved plot to ./results/Kd/v15/MMFE/fold1/embeddings/UMAP/img_train_val_embeddi